SILVER

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_products = spark.table("ecommerce.bronze.products")

silver_products = (
    bronze_products

    # ── 1. Cast correct types ──────────────────────────────────────
    .withColumn("price",        col("price").cast(DoubleType()))
    .withColumn("cost_price",   col("cost_price").cast(DoubleType()))
    .withColumn("stock_qty",    col("stock_qty").cast(IntegerType()))

    # ── 2. Clean strings ───────────────────────────────────────────
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category",     initcap(trim(col("category"))))
    .withColumn("brand",        initcap(trim(col("brand"))))

    # ── 3. Derived business columns ────────────────────────────────
    .withColumn("margin_amount",
                round(col("price") - col("cost_price"), 2))

    # FIX: zero-guard on cost_price to avoid division by zero
    .withColumn("margin_pct",
                when(col("cost_price") > 0,
                     round((col("price") - col("cost_price"))
                           / col("cost_price") * 100, 2))
                .otherwise(lit(None)))

    .withColumn("price_band",
                when(col("price") < 500,   lit("Budget"))
                .when(col("price") < 2000, lit("Mid-Range"))
                .when(col("price") < 5000, lit("Premium"))
                .otherwise(lit("Luxury")))

    .withColumn("stock_status",
                when(col("stock_qty") == 0,   lit("Out of Stock"))
                .when(col("stock_qty") < 10,  lit("Low Stock"))
                .when(col("stock_qty") < 100, lit("Medium Stock"))
                .otherwise(lit("High Stock")))

    # ── 4. Filter bad rows ─────────────────────────────────────────
    .filter(col("product_id").isNotNull())
    .filter(col("price") > 0)
    .filter(col("cost_price") > 0)
    .filter(col("price") >= col("cost_price"))   # price must be >= cost

    # ── 5. Deduplicate ─────────────────────────────────────────────
    .dropDuplicates(["product_id"])

    # ── 6. Audit ───────────────────────────────────────────────────
    .withColumn("silver_updated_at", current_timestamp())

    # ── 7. Select final columns ────────────────────────────────────
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "price",
        "cost_price",
        "margin_amount",
        "margin_pct",
        "price_band",
        "stock_qty",
        "stock_status",
        "silver_updated_at",
    )
)

(
    silver_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/products/")
    .saveAsTable("ecommerce.silver.products")
)

print(f"✅ silver.products → {silver_products.count():,} rows")
silver_products.show(5, truncate=False)

✅ silver.products → 8,000 rows
+----------+--------------------------+-----------+---------+--------+----------+-------------+----------+----------+---------+------------+--------------------------+
|product_id|product_name              |category   |brand    |price   |cost_price|margin_amount|margin_pct|price_band|stock_qty|stock_status|silver_updated_at         |
+----------+--------------------------+-----------+---------+--------+----------+-------------+----------+----------+---------+------------+--------------------------+
|P000001   |Mi Tablet V1              |Electronics|Mi       |5191.58 |3950.13   |1241.45      |31.43     |Luxury    |216      |High Stock  |2026-06-03 00:48:47.538049|
|P000002   |Adidas Cycling Helmet V2  |Sports     |Adidas   |10288.41|7098.4    |3190.01      |44.94     |Luxury    |398      |High Stock  |2026-06-03 00:48:47.538049|
|P000003   |Playskool Puzzle 1000pc V3|Toys       |Playskool|409.48  |216.3     |193.18       |89.31     |Budget    |17       |Me

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# 1. Read from Bronze
bronze_customers = spark.table("ecommerce.bronze.customers")

# 2. Transform
silver_customers = (
    bronze_customers
    
    # ── 1. Cast correct types & Handle empty strings early ─────────
    .withColumn("signup_date",  to_date(col("signup_date")))
    .withColumn("is_active",    col("is_active").cast(BooleanType()))
    # Convert empty/whitespace strings to proper Nulls
    .withColumn("phone",        when(trim(col("phone")) == "", lit(None)).otherwise(trim(col("phone"))))
    .withColumn("email",        when(trim(col("email")) == "", lit(None)).otherwise(lower(trim(col("email")))))
    .withColumn("full_name",    initcap(trim(col("full_name"))))
    .withColumn("city",         initcap(trim(col("city"))))
    .withColumn("state",        initcap(trim(col("state"))))

    # ── 2. Filter bad rows early (Saves compute for subsequent steps) ─
    .filter(col("customer_id").isNotNull())
    .filter(col("email").isNotNull())

    # ── 3. Deduplicate ─────────────────────────────────────────────
    .dropDuplicates(["customer_id"])

    # ── 4. Derived columns & Null Handling ─────────────────────────
    .withColumn("customer_tenure_days", datediff(current_date(), col("signup_date")))
    .withColumn("signup_year",  year(col("signup_date")))
    .withColumn("signup_month", month(col("signup_date")))
    .withColumn("email_domain", regexp_extract(col("email"), "@(.+)$", 1))
    
    # Now has_phone will be accurately False for both missing and empty numbers
    .withColumn("has_phone",    col("phone").isNotNull()) 
    .withColumn("phone",        coalesce(col("phone"), lit("NOT PROVIDED")))

    # ── 5. Audit column ────────────────────────────────────────────
    .withColumn("silver_updated_at", current_timestamp())

    # ── 6. Select final columns ────────────────────────────────────
    .select(
        "customer_id",
        "full_name",
        "email",
        "email_domain",
        "phone",
        "has_phone",
        "city",
        "state",
        "signup_date",
        "signup_year",
        "signup_month",
        "customer_tenure_days",
        "is_active",
        "silver_updated_at"
    )
)

# 3. Write to Silver
(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/customers/")
    .saveAsTable("ecommerce.silver.customers")
)

# 4. Efficiently print count using Delta metadata instead of re-computing
silver_count = spark.table("ecommerce.silver.customers").count()
print(f"✅ silver.customers → {silver_count:,} rows")
spark.table("ecommerce.silver.customers").show(5, truncate=False)

✅ silver.customers → 50,000 rows
+-----------+--------------+-----------------------+------------+-------------+---------+--------+-----------+-----------+-----------+------------+--------------------+---------+--------------------------+
|customer_id|full_name     |email                  |email_domain|phone        |has_phone|city    |state      |signup_date|signup_year|signup_month|customer_tenure_days|is_active|silver_updated_at         |
+-----------+--------------+-----------------------+------------+-------------+---------+--------+-----------+-----------+-----------+------------+--------------------+---------+--------------------------+
|C000006    |Simon Tata    |jshanker@example.com   |example.com |+913884969653|true     |Danapur |Punjab     |2022-05-09 |2022       |5           |1486                |true     |2026-06-03 01:01:25.417591|
|C000016    |Oni Virk      |advika35@example.org   |example.org |06247510799  |true     |Srinagar|Karnataka  |2023-07-25 |2023       |7        

In [0]:
bronze_orders    = spark.table("ecommerce.bronze.orders")
silver_customers = spark.table("ecommerce.silver.customers")
silver_products  = spark.table("ecommerce.silver.products")

silver_orders = (
    bronze_orders

    # ── 1. Cast correct types ──────────────────────────────────────
    .withColumn("order_date",    to_timestamp(col("order_date")))
    .withColumn("quantity",      col("quantity").cast(IntegerType()))
    .withColumn("total_amount",  col("total_amount").cast(DoubleType()))

    # ── 2. Clean strings ───────────────────────────────────────────
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("status",         initcap(trim(col("status"))))

    # ── 3. Derived date columns ────────────────────────────────────
    .withColumn("order_date_only", to_date(col("order_date")))
    .withColumn("order_year",      year(col("order_date")))
    .withColumn("order_month",     month(col("order_date")))
    .withColumn("order_day",       dayofmonth(col("order_date")))
    .withColumn("order_hour",      hour(col("order_date")))
    .withColumn("order_day_name",  date_format(col("order_date"), "EEEE"))
    .withColumn("order_quarter",   quarter(col("order_date")))

    # ── 4. Business flags ──────────────────────────────────────────
    .withColumn("is_delivered",
                when(col("status") == "Delivered", True).otherwise(False))
    .withColumn("is_cancelled",
                when(col("status") == "Cancelled", True).otherwise(False))
    .withColumn("is_returned",
                when(col("status") == "Returned",  True).otherwise(False))
    .withColumn("is_weekend",
                when(date_format(col("order_date"), "EEEE")
                     .isin("Saturday", "Sunday"), True).otherwise(False))

    # ── 5. Join customer details ───────────────────────────────────
    # FIX: drop the duplicate customer_id from silver_customers after join
    .join(
        silver_customers.select(
            col("customer_id").alias("_cust_id"),
            "full_name", "city", "state", "is_active", "customer_tenure_days"
        ),
        bronze_orders["customer_id"] == col("_cust_id"),
        how="left"
    )
    .drop("_cust_id")   # remove the aliased join key; bronze customer_id is kept

    # ── 6. Join product details ────────────────────────────────────
    .join(
        silver_products.select(
            "product_id", "product_name", "category",
            "brand", "price", "margin_pct", "price_band"
        ),
        on="product_id",
        how="left"
    )

    # ── 7. Filter bad rows ─────────────────────────────────────────
    .filter(col("order_id").isNotNull())
    .filter(col("total_amount") > 0)
    .filter(col("quantity") > 0)

    # ── 8. Deduplicate ─────────────────────────────────────────────
    .dropDuplicates(["order_id"])

    # ── 9. Audit ───────────────────────────────────────────────────
    .withColumn("silver_updated_at", current_timestamp())

    # ── 10. Select final columns ───────────────────────────────────
    .select(
        # Order keys
        "order_id",
        "customer_id",
        "product_id",
        # Customer enrichment
        "full_name",
        "city",
        "state",
        "is_active",
        "customer_tenure_days",
        # Product enrichment
        "product_name",
        "category",
        "brand",
        "price",
        "margin_pct",
        "price_band",
        # Order details
        "quantity",
        "total_amount",
        "payment_method",
        "status",
        # Date dimensions
        "order_date",
        "order_date_only",
        "order_year",
        "order_month",
        "order_quarter",
        "order_day",
        "order_hour",
        "order_day_name",
        # Flags
        "is_delivered",
        "is_cancelled",
        "is_returned",
        "is_weekend",
        # Audit
        "silver_updated_at",
    )
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/orders/")
    .saveAsTable("ecommerce.silver.orders")
)

print(f"✅ silver.orders → {silver_orders.count():,} rows")
silver_orders.show(5, truncate=False)

✅ silver.orders → 75,000 rows
+-----------+-----------+----------+-----------------+-----------+--------------+---------+--------------------+------------------------------------+--------------+-------------+--------+----------+----------+--------+------------+--------------+---------+-------------------+---------------+----------+-----------+-------------+---------+----------+--------------+------------+------------+-----------+----------+--------------------------+
|order_id   |customer_id|product_id|full_name        |city       |state         |is_active|customer_tenure_days|product_name                        |category      |brand        |price   |margin_pct|price_band|quantity|total_amount|payment_method|status   |order_date         |order_date_only|order_year|order_month|order_quarter|order_day|order_hour|order_day_name|is_delivered|is_cancelled|is_returned|is_weekend|silver_updated_at         |
+-----------+-----------+----------+-----------------+-----------+--------------+-----

In [0]:
bronze_clickstream = spark.table("ecommerce.bronze.clickstream")
silver_customers   = spark.table("ecommerce.silver.customers")
silver_products    = spark.table("ecommerce.silver.products")

silver_clickstream = (
    bronze_clickstream

    # ── 1. Cast correct types ──────────────────────────────────────
    # FIX: use coalesce so existing bronze event_ts is preferred
    .withColumn("event_ts",       coalesce(col("event_ts"), to_timestamp(col("timestamp"))))
    .withColumn("quantity",       col("quantity").cast(IntegerType()))
    .withColumn("unit_price",     col("unit_price").cast(DoubleType()))
    .withColumn("discount_pct",   col("discount_pct").cast(DoubleType()))
    .withColumn("amount",         col("amount").cast(DoubleType()))
    .withColumn("time_spent_sec", col("time_spent_sec").cast(IntegerType()))

    # ── 2. Clean strings ───────────────────────────────────────────
    .withColumn("event_type",     lower(trim(col("event_type"))))
    .withColumn("device_type",    lower(trim(col("device_type"))))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("page_url",       lower(trim(col("page_url"))))
    .withColumn("referrer",
                when(col("referrer").isNull(), lit("direct"))
                .otherwise(lower(trim(col("referrer")))))

    # ── 3. Derived date columns ────────────────────────────────────
    .withColumn("event_date",     to_date(col("event_ts")))
    .withColumn("event_year",     year(col("event_ts")))
    .withColumn("event_month",    month(col("event_ts")))
    .withColumn("event_hour",     hour(col("event_ts")))
    .withColumn("event_day_name", date_format(col("event_ts"), "EEEE"))
    .withColumn("is_weekend",
                when(date_format(col("event_ts"), "EEEE")
                     .isin("Saturday", "Sunday"), True).otherwise(False))

    # ── 4. Event type flags ────────────────────────────────────────
    .withColumn("is_page_view",
                when(col("event_type") == "page_view",    True).otherwise(False))
    .withColumn("is_product_view",
                when(col("event_type") == "product_view", True).otherwise(False))
    .withColumn("is_add_to_cart",
                when(col("event_type") == "add_to_cart",  True).otherwise(False))
    .withColumn("is_purchase",
                when(col("event_type") == "purchase",     True).otherwise(False))

    # ── 5. Join customer details ───────────────────────────────────
    # FIX: alias customer_id to avoid ambiguity with user_id
    .join(
        silver_customers.select(
            col("customer_id").alias("_cust_id"),
            "full_name", "city", "state", "is_active"
        ),
        bronze_clickstream["user_id"] == col("_cust_id"),
        how="left"
    )
    .drop("_cust_id")   # user_id remains as the clickstream key

    # ── 6. Join product details ────────────────────────────────────
    .join(
        silver_products.select(
            "product_id", "product_name",
            "category", "brand", "price_band"
        ),
        on="product_id",
        how="left"
    )

    # ── 7. Filter bad rows ─────────────────────────────────────────
    .filter(col("event_id").isNotNull())
    .filter(col("event_ts").isNotNull())

    # ── 8. Deduplicate ─────────────────────────────────────────────
    .dropDuplicates(["event_id"])

    # ── 9. Audit ───────────────────────────────────────────────────
    .withColumn("silver_updated_at", current_timestamp())

    # ── 10. Select final columns ───────────────────────────────────
    .select(
        # Event identity
        "event_id",
        "event_type",
        # User / session
        "user_id",
        "session_id",
        # Customer enrichment (null when anonymous)
        "full_name",
        "city",
        "state",
        "is_active",
        # Product enrichment (null for non-product events)
        "product_id",
        "product_name",
        "category",
        "brand",
        "price_band",
        # Order linkage
        "order_id",
        # Clickstream fields
        "page_url",
        "device_type",
        "referrer",
        "quantity",
        "unit_price",
        "discount_pct",
        "amount",
        "payment_method",
        "time_spent_sec",
        # Timestamps & date dims
        "event_ts",
        "event_date",
        "event_year",
        "event_month",
        "event_hour",
        "event_day_name",
        "is_weekend",
        # Event type flags
        "is_page_view",
        "is_product_view",
        "is_add_to_cart",
        "is_purchase",
        # Audit
        "silver_updated_at",
    )
)

(
    silver_clickstream.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/clickstream/")
    .saveAsTable("ecommerce.silver.clickstream")
)

print(f"✅ silver.clickstream → {silver_clickstream.count():,} rows")
silver_clickstream.show(5, truncate=False)

✅ silver.clickstream → 300,000 rows
+---------+------------+-------+----------+--------------+---------------+--------------+---------+----------+-----------------------------+--------+--------+----------+--------+----------------+-----------+--------+--------+----------+------------+------+--------------+--------------+-------------------+----------+----------+-----------+----------+--------------+----------+------------+---------------+--------------+-----------+--------------------------+
|event_id |event_type  |user_id|session_id|full_name     |city           |state         |is_active|product_id|product_name                 |category|brand   |price_band|order_id|page_url        |device_type|referrer|quantity|unit_price|discount_pct|amount|payment_method|time_spent_sec|event_ts           |event_date|event_year|event_month|event_hour|event_day_name|is_weekend|is_page_view|is_product_view|is_add_to_cart|is_purchase|silver_updated_at         |
+---------+------------+-------+----------

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_inventory = spark.table("ecommerce.bronze.inventory")
silver_products  = spark.table("ecommerce.silver.products")

silver_inventory = (
    bronze_inventory

    # ── 1. Cast correct types ──────────────────────────────────────
    .withColumn("available_qty",  col("available_qty").cast(IntegerType()))
    .withColumn("last_updated",   to_timestamp(col("last_updated")))

    # ── 2. Clean strings ───────────────────────────────────────────
    .withColumn("warehouse_id",   upper(trim(col("warehouse_id"))))

    # ── 3. Join product details ────────────────────────────────────
    # FIX: Optimized using broadcast join
    .join(
        broadcast(silver_products.select(
            "product_id", "product_name",
            "category", "brand",
            "price", "stock_status"
        )),
        on="product_id",
        how="left"
    )

    # ── 4. Derived stock health columns ────────────────────────────
    .withColumn("stock_health",
                when(col("available_qty") == 0,   lit("Dead Stock"))
                .when(col("available_qty") < 10,  lit("Critical"))
                .when(col("available_qty") < 50,  lit("Low"))
                .when(col("available_qty") < 200, lit("Healthy"))
                .otherwise(lit("Overstocked")))

    .withColumn("needs_restock",
                when(col("available_qty") < 10, True).otherwise(False))

    .withColumn("inventory_value",
                round(col("available_qty") * col("price"), 2))

    .withColumn("days_since_update",
                datediff(current_date(), col("last_updated")))

    .withColumn("is_stale",
                when(col("days_since_update") > 30, True).otherwise(False))

    # ── 5. Filter bad rows ─────────────────────────────────────────
    .filter(col("product_id").isNotNull())
    .filter(col("warehouse_id").isNotNull())
    .filter(col("available_qty") >= 0)

    # ── 6. Deduplicate ─────────────────────────────────────────────
    .dropDuplicates(["product_id", "warehouse_id"])

    # ── 7. Audit ───────────────────────────────────────────────────
    .withColumn("silver_updated_at", current_timestamp())

    # ── 8. Select final columns ────────────────────────────────────
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "warehouse_id",
        "available_qty",
        "stock_health",
        "needs_restock",
        "inventory_value",
        "price",
        "last_updated",
        "days_since_update",
        "is_stale",
        "silver_updated_at",
    )
)

(
    silver_inventory.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/inventory/")
    .saveAsTable("ecommerce.silver.inventory")
)

print(f"✅ silver.inventory → {silver_inventory.count():,} rows")
silver_inventory.show(5, truncate=False)

✅ silver.inventory → 15,899 rows
+----------+-------------------------+--------------+-----------+------------+-------------+------------+-------------+---------------+--------+-------------------+-----------------+--------+--------------------------+
|product_id|product_name             |category      |brand      |warehouse_id|available_qty|stock_health|needs_restock|inventory_value|price   |last_updated       |days_since_update|is_stale|silver_updated_at         |
+----------+-------------------------+--------------+-----------+------------+-------------+------------+-------------+---------------+--------+-------------------+-----------------+--------+--------------------------+
|P000016   |Havells Air Fryer V16    |Home & Kitchen|Havells    |WH15        |281          |Overstocked |false        |2302365.07     |8193.47 |2024-05-28 22:00:00|736              |true    |2026-06-03 01:05:36.406425|
|P000048   |Puma Resistance Bands V48|Sports        |Puma       |WH13        |273          

In [0]:
from pyspark.sql.functions import *

silver_inventory = spark.table("ecommerce.silver.inventory")
silver_orders    = spark.table("ecommerce.silver.orders")

# ── Step 1: Demand summary per product from orders ────────────────────────
product_demand = (
    silver_orders
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_qty_ordered"),
        count("order_id").alias("total_orders"),
        sum("total_amount").alias("total_revenue_generated"),
        avg("total_amount").alias("avg_order_value"),

        sum(when(col("is_delivered") == True, col("quantity"))).alias("qty_delivered"),
        sum(when(col("is_delivered") == True, col("total_amount"))).alias("revenue_delivered"),

        sum(when(col("is_cancelled") == True, col("quantity"))).alias("qty_cancelled"),
        sum(when(col("is_returned")  == True, col("quantity"))).alias("qty_returned"),

        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date"),

        countDistinct("customer_id").alias("unique_customers_ordered"),
    )
    .withColumn("qty_delivered", coalesce(col("qty_delivered"), lit(0)))
    .withColumn("qty_cancelled", coalesce(col("qty_cancelled"), lit(0)))
    .withColumn("qty_returned",  coalesce(col("qty_returned"),  lit(0)))
)

# ── Step 2: Total stock per product across all warehouses ─────────────────
total_stock = (
    silver_inventory
    .groupBy("product_id")
    .agg(
        sum("available_qty").alias("total_stock_all_warehouses"),
        count("warehouse_id").alias("warehouse_count"),
        sum("inventory_value").alias("total_inventory_value"),
        collect_list("warehouse_id").alias("warehouse_list"),
    )
)

# ── Step 3: Join inventory + demand ───────────────────────────────────────
inventory_orders = (
    silver_inventory

    .join(product_demand, on="product_id", how="left")
    .join(total_stock,    on="product_id", how="left")

    # ── Fill nulls for products with no orders ─────────────────────
    .withColumn("total_qty_ordered",
        coalesce(col("total_qty_ordered"), lit(0)))
    .withColumn("total_orders",
        coalesce(col("total_orders"), lit(0)))
    .withColumn("total_revenue_generated",
        coalesce(col("total_revenue_generated"), lit(0.0)))

    # ── Stock to demand ratio ──────────────────────────────────────
    .withColumn("stock_to_demand_ratio",
        when(col("total_qty_ordered") > 0,
             round(col("total_stock_all_warehouses")
                   / col("total_qty_ordered"), 2))
        .otherwise(lit(None)))

    # ── Demand health classification ───────────────────────────────
    .withColumn("demand_health",
        when(col("total_orders") == 0,                      lit("No Demand"))
        .when(col("stock_to_demand_ratio") < 0.25,          lit("Critical — Restock Now"))
        .when(col("stock_to_demand_ratio") < 0.5,           lit("Low Stock vs Demand"))
        .when(col("stock_to_demand_ratio").between(0.5, 2), lit("Balanced"))
        .when(col("stock_to_demand_ratio") > 5,             lit("Overstocked"))
        .otherwise(lit("Monitor")))

    # ── Days of stock remaining (rough estimate) ───────────────────
    # FIX: replaced -- SQL comment with # Python comment
    .withColumn("days_of_stock_remaining",
        when(
            (col("total_qty_ordered") > 0) &
            (col("total_stock_all_warehouses") > 0),
            round(
                col("total_stock_all_warehouses")
                / (col("total_qty_ordered") / 730.0)   # orders spread over ~2 years
            , 0)
        ).otherwise(lit(None)))

    .withColumn("silver_updated_at", current_timestamp())

    # ── Final column selection ─────────────────────────────────────
    # FIX: completed the truncated select + added missing columns
    .select(
        # Product identity
        "product_id",
        "product_name",
        "category",
        "brand",
        "price",
        # Per-warehouse inventory
        "warehouse_id",
        "available_qty",
        "stock_health",
        "needs_restock",
        "inventory_value",
        "is_stale",
        "last_updated",
        "days_since_update",
        # Totals across all warehouses
        "total_stock_all_warehouses",
        "warehouse_count",
        "total_inventory_value",
        "warehouse_list",
        # Demand from orders
        "total_qty_ordered",
        "total_orders",
        "total_revenue_generated",
        "avg_order_value",
        "qty_delivered",
        "revenue_delivered",
        "qty_cancelled",
        "qty_returned",
        "first_order_date",
        "last_order_date",
        "unique_customers_ordered",
        # Stock vs demand
        "stock_to_demand_ratio",
        "demand_health",
        "days_of_stock_remaining",
        # Audit
        "silver_updated_at",
    )
)

(
    inventory_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/inventory_orders/")
    .saveAsTable("ecommerce.silver.inventory_orders")
)

print(f"✅ silver.inventory_orders → {inventory_orders.count():,} rows")
inventory_orders.show(5, truncate=False)

✅ silver.inventory_orders → 15,899 rows
+----------+-------------------------+--------------+-----------+--------+------------+-------------+------------+-------------+---------------+--------+-------------------+-----------------+--------------------------+---------------+---------------------+------------------+-----------------+------------+-----------------------+------------------+-------------+------------------+-------------+------------+-------------------+-------------------+------------------------+---------------------+-------------------+-----------------------+--------------------------+
|product_id|product_name             |category      |brand      |price   |warehouse_id|available_qty|stock_health|needs_restock|inventory_value|is_stale|last_updated       |days_since_update|total_stock_all_warehouses|warehouse_count|total_inventory_value|warehouse_list    |total_qty_ordered|total_orders|total_revenue_generated|avg_order_value   |qty_delivered|revenue_delivered |qty_cancel

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_customers   = spark.table("ecommerce.silver.customers")
silver_orders      = spark.table("ecommerce.silver.orders")
silver_clickstream = spark.table("ecommerce.silver.clickstream")

# ── Step 1: Order behaviour per customer ──────────────────────────────────
customer_orders_base = (
    silver_orders
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_items_ordered"),
        sum("total_amount").alias("total_spend"),
        avg("total_amount").alias("avg_order_value"),
        max("total_amount").alias("max_order_value"),
        min("total_amount").alias("min_order_value"),
        sum(when(col("is_delivered") == True, lit(1)).otherwise(lit(0))).alias("orders_delivered"),
        sum(when(col("is_cancelled") == True, lit(1)).otherwise(lit(0))).alias("orders_cancelled"),
        sum(when(col("is_returned")  == True, lit(1)).otherwise(lit(0))).alias("orders_returned"),
        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date"),
        sum(when(col("is_weekend") == True, lit(1)).otherwise(lit(0))).alias("weekend_orders"),
    )
    .withColumn("days_since_last_order",
        datediff(current_date(), col("last_order_date")))
    .withColumn("cancellation_rate",
        round(col("orders_cancelled") / col("total_orders") * 100, 2))
    .withColumn("return_rate",
        round(col("orders_returned") / col("total_orders") * 100, 2))
    .withColumn("weekend_order_rate",
        round(col("weekend_orders") / col("total_orders") * 100, 2))
)

# FIX: use ranked window to get TRUE most-frequent category and payment method
# instead of unreliable first()
cat_rank_window = Window.partitionBy("customer_id").orderBy(desc("_cnt"))
fav_category_df = (
    silver_orders.groupBy("customer_id", "category").agg(count("*").alias("_cnt"))
    .withColumn("_rn", row_number().over(cat_rank_window))
    .filter(col("_rn") == 1)
    .select("customer_id", col("category").alias("fav_category"))
)

pay_rank_window = Window.partitionBy("customer_id").orderBy(desc("_cnt"))
fav_payment_df = (
    silver_orders.groupBy("customer_id", "payment_method").agg(count("*").alias("_cnt"))
    .withColumn("_rn", row_number().over(pay_rank_window))
    .filter(col("_rn") == 1)
    .select("customer_id", col("payment_method").alias("fav_payment_method"))
)

customer_orders = (
    customer_orders_base
    .join(fav_category_df, on="customer_id", how="left")
    .join(fav_payment_df,  on="customer_id", how="left")
)

# ── Step 2: Clickstream behaviour per customer ────────────────────────────
customer_clicks = (
    silver_clickstream
    .groupBy("user_id")
    .agg(
        countDistinct("session_id").alias("total_sessions"),
        count("event_id").alias("total_events"),
        sum(when(col("is_page_view")    == True, lit(1)).otherwise(lit(0))).alias("total_page_views"),
        sum(when(col("is_product_view") == True, lit(1)).otherwise(lit(0))).alias("total_product_views"),
        sum(when(col("is_add_to_cart")  == True, lit(1)).otherwise(lit(0))).alias("total_add_to_carts"),
        sum(when(col("is_purchase")     == True, lit(1)).otherwise(lit(0))).alias("total_purchase_events"),
        first("device_type").alias("preferred_device"),
        first("referrer").alias("preferred_referrer"),
        sum(when(col("time_spent_sec").isNotNull(), col("time_spent_sec"))).alias("total_time_spent_sec"),
        max("event_ts").alias("last_seen_ts"),
        min("event_ts").alias("first_seen_ts"),
    )
    .withColumnRenamed("user_id", "customer_id")
    .withColumn("avg_events_per_session",
        round(col("total_events") / col("total_sessions"), 2))
    .withColumn("abandoned_carts",
        (col("total_add_to_carts") - col("total_purchase_events")))
    .withColumn("cart_conversion_rate",
        when(col("total_add_to_carts") > 0,
             round(col("total_purchase_events") / col("total_add_to_carts") * 100, 2))
        .otherwise(lit(0.0)))
)

# ── Step 3: Join everything to customers ──────────────────────────────────
customer_360 = (
    silver_customers

    .join(customer_orders, on="customer_id", how="left")
    .join(customer_clicks, on="customer_id", how="left")

    # ── Fill nulls for customers with no activity ──────────────────
    .withColumn("total_orders",    coalesce(col("total_orders"),    lit(0)))
    .withColumn("total_spend",     coalesce(col("total_spend"),     lit(0.0)))
    .withColumn("total_sessions",  coalesce(col("total_sessions"),  lit(0)))
    .withColumn("total_events",    coalesce(col("total_events"),    lit(0)))
    .withColumn("abandoned_carts", coalesce(col("abandoned_carts"), lit(0)))

    # ── Customer Value Segment ─────────────────────────────────────
    .withColumn("customer_value_segment",
        when(col("total_spend") >= 50000, lit("Platinum"))
        .when(col("total_spend") >= 20000, lit("Gold"))
        .when(col("total_spend") >= 5000,  lit("Silver"))
        .when(col("total_spend") >  0,     lit("Bronze"))
        .otherwise(lit("No Purchase")))

    # ── Churn Risk ─────────────────────────────────────────────────
    # FIX: check Never Purchased FIRST before null days_since_last_order comparisons
    .withColumn("churn_risk",
        when(col("total_orders") == 0,            lit("Never Purchased"))
        .when(col("days_since_last_order") > 180, lit("High Risk"))
        .when(col("days_since_last_order") > 90,  lit("Medium Risk"))
        .when(col("days_since_last_order") > 30,  lit("Low Risk"))
        .otherwise(lit("Active")))

    # ── Customer Loyalty Tag ───────────────────────────────────────
    .withColumn("loyalty_tag",
        when(col("total_orders") >= 20, lit("Champion"))
        .when(col("total_orders") >= 10, lit("Loyal"))
        .when(col("total_orders") >= 5,  lit("Promising"))
        .when(col("total_orders") >= 1,  lit("New Customer"))
        .otherwise(lit("Visitor Only")))

    .withColumn("silver_updated_at", current_timestamp())

    # ── Final column selection ─────────────────────────────────────
    .select(
        "customer_id", "full_name", "email", "email_domain",
        "phone", "has_phone", "city", "state",
        "signup_date", "signup_year", "signup_month", "customer_tenure_days", "is_active",
        "total_orders", "total_items_ordered", "total_spend",
        "avg_order_value", "max_order_value", "min_order_value",
        "orders_delivered", "orders_cancelled", "orders_returned",
        "cancellation_rate", "return_rate",
        "fav_category", "fav_payment_method",
        "weekend_orders", "weekend_order_rate",
        "first_order_date", "last_order_date", "days_since_last_order",
        "total_sessions", "total_events",
        "total_page_views", "total_product_views",
        "total_add_to_carts", "total_purchase_events",
        "abandoned_carts", "cart_conversion_rate",
        "preferred_device", "preferred_referrer",
        "total_time_spent_sec", "avg_events_per_session",
        "first_seen_ts", "last_seen_ts",
        "customer_value_segment", "churn_risk", "loyalty_tag",
        "silver_updated_at",
    )
)

(
    customer_360.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "s3://ecommerce-lakehouse-databricks/silver/customer_360/")
    .saveAsTable("ecommerce.silver.customer_360")
)

print(f"✅ silver.customer_360 → {customer_360.count():,} rows")
customer_360.show(5, truncate=False)

✅ silver.customer_360 → 50,000 rows
+-----------+--------------+--------------------------+------------+-------------+---------+----------+-----------+-----------+-----------+------------+--------------------+---------+------------+-------------------+-----------+-----------------+---------------+---------------+----------------+----------------+---------------+-----------------+-----------+------------+------------------+--------------+------------------+-------------------+-------------------+---------------------+--------------+------------+----------------+-------------------+------------------+---------------------+---------------+--------------------+----------------+------------------+--------------------+----------------------+-------------------+-------------------+----------------------+---------------+------------+-------------------------+
|customer_id|full_name     |email                     |email_domain|phone        |has_phone|city      |state      |signup_date|signup_ye

In [0]:
# Retrieve the list of tables
silver_tables = spark.catalog.listTables("ecommerce.silver")

# Display the tables and their schemas
for table in silver_tables:
    print("=" * 60)
    print(f"📋 TABLE: {table.name} ({table.tableType})")
    print("=" * 60)
    
    try:
        # Load the table profile to read its schema
        df = spark.table(f"ecommerce.silver.{table.name}")
        
        # Loop through and print each column name and its data type
        for field in df.schema:
            print(f"  ▪️ {field.name:<30} : {field.dataType.simpleString()}")
            
    except Exception as e:
        print(f"  ❌ Could not retrieve schema: {e}")
        
    print("\n")

📋 TABLE: clickstream (EXTERNAL)
  ▪️ event_id                       : string
  ▪️ event_type                     : string
  ▪️ user_id                        : string
  ▪️ session_id                     : string
  ▪️ full_name                      : string
  ▪️ city                           : string
  ▪️ state                          : string
  ▪️ is_active                      : boolean
  ▪️ product_id                     : string
  ▪️ product_name                   : string
  ▪️ category                       : string
  ▪️ brand                          : string
  ▪️ price_band                     : string
  ▪️ order_id                       : string
  ▪️ page_url                       : string
  ▪️ device_type                    : string
  ▪️ referrer                       : string
  ▪️ quantity                       : int
  ▪️ unit_price                     : double
  ▪️ discount_pct                   : double
  ▪️ amount                         : double
  ▪️ payment_method      

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce.gold;
SHOW TABLES IN ecommerce.silver;

In [0]:
%sql
-- Total delivered revenue across all time
SELECT
    SUM(total_revenue)              AS total_delivered_revenue,
    SUM(total_orders)               AS total_delivered_orders,
    ROUND(AVG(avg_order_value), 2)  AS overall_avg_order_value
FROM ecommerce.gold.monthly_summary

In [0]:
%sql
-- Top 5 categories by revenue
SELECT category, SUM(total_revenue) AS revenue
FROM ecommerce.gold.revenue_by_category
GROUP BY category
ORDER BY revenue DESC
LIMIT 5